# CAPTCHA CNN Training - MITS Gwalior Result Scraper

Train a Convolutional Neural Network to recognize 5-character alphanumeric CAPTCHAs
from the MITS Gwalior university result portal.

**Architecture**: Multi-head CNN with 4 Conv blocks + 5 independent classification heads

**Dataset**: 50,000 synthetic CAPTCHAs generated on-the-fly

---
**Make sure to enable GPU**: Runtime -> Change runtime type -> T4 GPU

## 1. Setup & Imports

In [ ]:
# Install dependencies (most are pre-installed on Colab)
!pip install -q torch torchvision Pillow matplotlib tqdm

In [ ]:
import os
import random
import string
import time
import json

import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Configuration

In [ ]:
# Character set
CHARSET = string.ascii_uppercase + string.digits  # A-Z + 0-9
NUM_CLASSES = len(CHARSET)  # 36
CAPTCHA_LENGTH = 5
IMG_WIDTH = 200
IMG_HEIGHT = 80

# Mappings
CHAR_TO_IDX = {ch: idx for idx, ch in enumerate(CHARSET)}
IDX_TO_CHAR = {idx: ch for idx, ch in enumerate(CHARSET)}

# Training config
NUM_TRAIN = 50000
NUM_VAL = 5000
BATCH_SIZE = 64
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3
PATIENCE = 7

print(f'Character set: {CHARSET}')
print(f'Classes: {NUM_CLASSES}')
print(f'CAPTCHA length: {CAPTCHA_LENGTH}')
print(f'Image size: {IMG_WIDTH}x{IMG_HEIGHT}')

## 3. Synthetic CAPTCHA Generator

Generates CAPTCHAs that mimic the MITS portal style:
- Bold black text on white background
- 5 uppercase alphanumeric characters
- Slight noise and character jitter

In [ ]:
def get_font(size=40):
    """Get a bold font (Colab-compatible)."""
    font_paths = [
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
        '/usr/share/fonts/truetype/freefont/FreeSansBold.ttf',
    ]
    for path in font_paths:
        if os.path.exists(path):
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()


def generate_captcha_image(text=None, add_noise=True):
    """Generate a single CAPTCHA image."""
    if text is None:
        text = ''.join(random.choices(CHARSET, k=CAPTCHA_LENGTH))

    img = Image.new('RGB', (IMG_WIDTH, IMG_HEIGHT), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)

    font_size = random.randint(36, 44)
    font = get_font(font_size)

    # Calculate character positions
    char_widths = []
    for ch in text:
        bbox = font.getbbox(ch)
        char_widths.append(bbox[2] - bbox[0])

    spacing = random.randint(2, 6)
    total_width = sum(char_widths) + spacing * (CAPTCHA_LENGTH - 1)
    x_cursor = (IMG_WIDTH - total_width) // 2

    for i, ch in enumerate(text):
        y_offset = random.randint(-4, 4)
        y_pos = (IMG_HEIGHT - font_size) // 2 + y_offset
        r, g, b = random.randint(0, 30), random.randint(0, 30), random.randint(0, 30)
        draw.text((x_cursor, y_pos), ch, fill=(r, g, b), font=font)
        x_cursor += char_widths[i] + spacing

    if add_noise:
        img_array = np.array(img)
        for _ in range(random.randint(50, 200)):
            x, y = random.randint(0, IMG_WIDTH-1), random.randint(0, IMG_HEIGHT-1)
            img_array[y, x] = [0,0,0] if random.random() > 0.5 else [200,200,200]
        img = Image.fromarray(img_array)

    if random.random() < 0.3:
        img = img.filter(ImageFilter.GaussianBlur(radius=0.5))

    return img, text


# Visualize samples
fig, axes = plt.subplots(2, 5, figsize=(15, 5))
for ax in axes.flat:
    img, text = generate_captcha_image()
    ax.imshow(img)
    ax.set_title(text, fontsize=12, fontweight='bold')
    ax.axis('off')
plt.suptitle('Synthetic CAPTCHA Samples', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. PyTorch Dataset

In [ ]:
class SyntheticCaptchaDataset(Dataset):
    def __init__(self, num_samples, add_noise=True):
        self.num_samples = num_samples
        self.add_noise = add_noise
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5]),
        ])
        self.texts = [''.join(random.choices(CHARSET, k=CAPTCHA_LENGTH))
                      for _ in range(num_samples)]

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        text = self.texts[idx]
        img, _ = generate_captcha_image(text=text, add_noise=self.add_noise)
        img_tensor = self.transform(img)
        label = torch.tensor([CHAR_TO_IDX[ch] for ch in text], dtype=torch.long)
        return img_tensor, label


# Create datasets
print(f'Creating training dataset ({NUM_TRAIN:,} samples)...')
train_dataset = SyntheticCaptchaDataset(NUM_TRAIN, add_noise=True)
print(f'Creating validation dataset ({NUM_VAL:,} samples)...')
val_dataset = SyntheticCaptchaDataset(NUM_VAL, add_noise=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

# Verify
images, labels = next(iter(train_loader))
print(f'Batch shape: {images.shape}')
print(f'Labels shape: {labels.shape}')
print(f'Sample label: {labels[0]} -> {chr(34).join([IDX_TO_CHAR[l.item()] for l in labels[0]])}')

## 5. CNN Model Architecture

Multi-head CNN:
- **Feature Extractor**: 4 Conv blocks (Conv2d -> BatchNorm -> ReLU -> MaxPool)
- **Classifier**: Shared FC layers + 5 independent heads (one per character position)
- **Output**: 5 tensors of (batch, 36) - each head classifies one character

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        return self.pool(F.relu(self.bn(self.conv(x)), inplace=True))


class CaptchaCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, captcha_length=CAPTCHA_LENGTH, dropout=0.3):
        super().__init__()
        self.captcha_length = captcha_length

        # Feature extractor
        # (1, 80, 200) -> (32, 40, 100) -> (64, 20, 50) -> (128, 10, 25) -> (256, 5, 12)
        self.features = nn.Sequential(
            ConvBlock(1, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
        )

        flat_size = 256 * 5 * 12  # 15,360

        # Shared FC
        self.shared_fc = nn.Sequential(
            nn.Linear(flat_size, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )

        # 5 classification heads
        self.heads = nn.ModuleList([nn.Linear(512, num_classes) for _ in range(captcha_length)])

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.shared_fc(x)
        return [head(x) for head in self.heads]


# Create model
model = CaptchaCNN().to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {num_params:,}')
print(f'\nArchitecture:\n{model}')

## 6. Training Loop

In [ ]:
def compute_accuracy(outputs, labels):
    batch_size = labels.size(0)
    correct_chars = 0
    preds = []
    for i, out in enumerate(outputs):
        _, pred = out.max(dim=1)
        preds.append(pred)
        correct_chars += (pred == labels[:, i]).sum().item()
    preds = torch.stack(preds, dim=1)
    correct_seqs = (preds == labels).all(dim=1).sum().item()
    return correct_chars / (batch_size * CAPTCHA_LENGTH), correct_seqs / batch_size


# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# Training history
history = {'train_loss': [], 'val_loss': [], 'train_char_acc': [],
           'val_char_acc': [], 'train_seq_acc': [], 'val_seq_acc': []}

best_val_loss = float('inf')
patience_counter = 0

print(f'Training for {NUM_EPOCHS} epochs on {device}...')
print(f'{"Epoch":>5} | {"Train Loss":>10} | {"Val Loss":>10} | '
      f'{"Train Char%":>11} | {"Val Char%":>10} | '
      f'{"Train Seq%":>10} | {"Val Seq%":>9} | {"LR":>10} | {"Time":>6}')
print('-' * 105)

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()

    # --- Training ---
    model.train()
    train_loss, train_char, train_seq, n_batches = 0, 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = sum(criterion(outputs[i], labels[:, i]) for i in range(CAPTCHA_LENGTH))
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        ca, sa = compute_accuracy(outputs, labels)
        train_char += ca; train_seq += sa; n_batches += 1
    train_loss /= n_batches; train_char /= n_batches; train_seq /= n_batches

    # --- Validation ---
    model.eval()
    val_loss, val_char, val_seq, n_batches = 0, 0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = sum(criterion(outputs[i], labels[:, i]) for i in range(CAPTCHA_LENGTH))
            val_loss += loss.item()
            ca, sa = compute_accuracy(outputs, labels)
            val_char += ca; val_seq += sa; n_batches += 1
    val_loss /= n_batches; val_char /= n_batches; val_seq /= n_batches

    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']
    elapsed = time.time() - start

    # Record
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_char_acc'].append(train_char)
    history['val_char_acc'].append(val_char)
    history['train_seq_acc'].append(train_seq)
    history['val_seq_acc'].append(val_seq)

    marker = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss, 'val_char_acc': val_char, 'val_seq_acc': val_seq,
            'config': {'num_classes': NUM_CLASSES, 'captcha_length': CAPTCHA_LENGTH,
                       'img_height': IMG_HEIGHT, 'img_width': IMG_WIDTH},
        }, 'captcha_cnn.pth')
        marker = ' << saved'
    else:
        patience_counter += 1

    print(f'{epoch:5d} | {train_loss:10.4f} | {val_loss:10.4f} | '
          f'{train_char:10.4f} | {val_char:10.4f} | '
          f'{train_seq:10.4f} | {val_seq:9.4f} | {lr:10.6f} | {elapsed:5.1f}s{marker}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

print(f'\nTraining complete! Best val loss: {best_val_loss:.4f}')

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-', label='Train')
axes[0].plot(epochs_range, history['val_loss'], 'r-', label='Validation')
axes[0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_char_acc'], 'b-', label='Train')
axes[1].plot(epochs_range, history['val_char_acc'], 'r-', label='Validation')
axes[1].set_title('Per-Character Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, history['train_seq_acc'], 'b-', label='Train')
axes[2].plot(epochs_range, history['val_seq_acc'], 'r-', label='Validation')
axes[2].set_title('Full Sequence Accuracy', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 8. Test the Model

In [ ]:
# Load best model
checkpoint = torch.load('captcha_cnn.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Best model from epoch {checkpoint['epoch']}")
print(f"Val char acc: {checkpoint['val_char_acc']:.4f}")
print(f"Val seq acc:  {checkpoint['val_seq_acc']:.4f}")

# Test on new samples
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

fig, axes = plt.subplots(3, 5, figsize=(18, 8))
correct = 0
total = 15

for ax in axes.flat:
    img, true_text = generate_captcha_image(add_noise=True)
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        pred_text = ''.join([IDX_TO_CHAR[out.argmax(1).item()] for out in outputs])

    is_correct = pred_text == true_text
    correct += int(is_correct)

    ax.imshow(img)
    color = 'green' if is_correct else 'red'
    ax.set_title(f'True: {true_text}\nPred: {pred_text}', fontsize=10,
                 color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle(f'Test Results: {correct}/{total} correct ({correct/total*100:.0f}%)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Download Model

Download the trained model to use locally with the scraper.

Place the downloaded `captcha_cnn.pth` file in:
```
BHEL PROJECT/captcha_model/saved_models/captcha_cnn.pth
```

In [ ]:
# Check model file
import os
model_size = os.path.getsize('captcha_cnn.pth') / (1024 * 1024)
print(f'Model file size: {model_size:.1f} MB')

# Download (Google Colab)
try:
    from google.colab import files
    files.download('captcha_cnn.pth')
    files.download('training_curves.png')
    print('\nFiles downloaded!')
    print('Place captcha_cnn.pth in: captcha_model/saved_models/')
except ImportError:
    print('Not running on Colab. Model saved locally as captcha_cnn.pth')